<a href="https://colab.research.google.com/github/valdsab/AFFiNE/blob/canary/Building_a_Multi_Agent_Research_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG for financial documents

In this notebook you'll learn how to build an AI agent that can perform multi-step research over a collection of 10K filings for Adobe and also generate a report.

We use LlamaCloud to index the bank of documents and agent abstractions to perform agentic RAG research along with the other steps in the workflow (saving notes and report writing).

In [ ]:
# install needed dependencies
!pip install -qU openai llama-index-core llama-index-embeddings-openai llama-index-llms-openai "llama-index-indices-managed-llama-cloud>=0.7.0" llama-cloud-services

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 60.2 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["LLAMA_CLOUD_API_KEY"] = userdata.get('LLAMA_CLOUD_API_KEY')

## Specify the data

If you want to have a fully functional Agentic RAG, you **always** have to start with the data: in our case, since we want to build an agentic RAG application for financial documents, we will need to get some publicly available financial report from some company out there - for this example, we chose **Adobe**, fetching their data from [AnnualReports.com](https://www.annualreports.com/Company/adobe-systems-inc).

In [ ]:
urls = [
    "https://www.annualreports.com/HostedData/AnnualReportArchive/a/NASDAQ_ADBE_2023.pdf",
    "https://www.annualreports.com/HostedData/AnnualReportArchive/a/NASDAQ_ADBE_2022.pdf",
    "https://www.annualreports.com/HostedData/AnnualReportArchive/a/NASDAQ_ADBE_2021.pdf",
]

## Index the data

It is not enough to *have* the data, we also need to pre-process them in order to be then ready for retrieval.

Using a [LlamaCloud Index](https://docs.cloud.llamaindex.ai/llamacloud/getting_started), we can configure
1. LlamaParse for parsing
2. OpenAI for embeddings
3. Upload directly from URLs

In [ ]:
from llama_cloud import LlamaParseParameters, PipelineEmbeddingConfig_OpenaiEmbedding
from llama_index.indices.managed.llama_cloud import LlamaCloudIndex

embedding_config = PipelineEmbeddingConfig_OpenaiEmbedding.parse_obj({
  "type": "OPENAI_EMBEDDING",
  "component": {
    "model_name": "text-embedding-3-large",
    "api_key": os.environ["OPENAI_API_KEY"]
  }
})

index = await LlamaCloudIndex.acreate_index(
    name="adobe_financial_data",
    # project_name="jerry_project", # TODO: REPLACE
    # organization_id="43b88c8f-e488-46f6-9013-698e3d2e374a",  # TODO: REPLACE
    embedding_config=embedding_config,
    llama_parse_parameters=LlamaParseParameters(
        parse_mode="parse_document_with_agent",
        model="anthropic-sonnet-4.0",
    )
)

In [ ]:
file_ids = []
for url in urls:
  file_ids.append(
      await index.aupload_file_from_url(
        file_name=url.split("/")[-1],
        url=url,
        wait_for_ingestion=False
    )
  )

In [ ]:
await index.await_for_completion(
    file_ids=file_ids,
    sleep_interval=5.0,
    verbose=True,
)

Loading files
File ingestion finished for 0b4e7e1d-9a85-4164-ab17-594893e887aa
File ingestion finished for 49ba1f79-70e2-496d-973d-304f1920c2d0
File ingestion finished for b19cf32d-097f-47c9-a7ad-f625f3d4a0b5
Done!
Syncing pipeline 23ad0694-32d0-410e-9927-8020bae6fe6e
..Done!


ManagedIngestionStatusResponse(job_id='e5622bd0-9dc1-4555-967a-2f83514cd04e', deployment_date=datetime.datetime(2025, 5, 29, 14, 43, 55, 699000, tzinfo=datetime.timezone.utc), status=<ManagedIngestionStatus.SUCCESS: 'SUCCESS'>, error=None, effective_at=datetime.datetime(2025, 5, 29, 14, 44, 45, 538213))

## Connect to the index

Now that we've created the index, let's connect to it and test its retrieval capacities!

In [ ]:
# define a default LLM
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

llm = OpenAI(
    model="gpt-4.1",
)

Settings.llm = llm

In [ ]:
# connect to the index
from llama_index.indices.managed.llama_cloud import LlamaCloudIndex
from llama_index.core.llms import ChatMessage
from llama_index.core.prompts import RichPromptTemplate

index = LlamaCloudIndex(
    name="adobe_financial_data",
    project_name="jerry_project", # TODO: replace
    organization_id="43b88c8f-e488-46f6-9013-698e3d2e374a", # TODO: replace

)
query_engine = index.as_query_engine(
    dense_similarity_top_k=6,
    sparse_similarity_top_k=6,
    rerank_top_n=3,
    files_top_k=1,
    retrieval_mode="auto_routed",
)

# index.as_retriever(...)

In [ ]:
# test the retrieval capacities
from IPython.display import Markdown, display

response = await query_engine.aquery("Who are Adobe’s main competitors in the Digital Media space?")
display(Markdown(response.response))

Adobe’s main competitors in the Digital Media space include large, established software and AI companies, device, hardware, and camera manufacturers (especially those integrating digital media software), proprietary and open-source web-authoring tools, mobile-first apps, web-native tools and platforms, social media platforms that offer digital media and editing capabilities, providers of stock content, digital document creation, storage, collaboration and signing providers, and operating system developers that integrate digital media document viewers and markup features with their systems. The competitive landscape also includes a variety of point offerings, free products, downloadable apps, and other specialized products and services.

That is already a good answer, but we posed a simple question!

If we want our system to be able to answer advanced, analytical questions, we should make it such that our questions are chunked into **multiple steps**, and for each steps the query engine retrieves information, combining everything at the end.

Typically, you would do this with an agent, that can then invoke our query engine multiple times to get results.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

async def query_financial_data(atomic_question: str) -> str:
  """Useful for getting answers to atomic and direct natural language queries about data concerning Adobe's finances."""
  response = await query_engine.aquery(atomic_question)
  return str(response)

query_agent = FunctionAgent(
    tools=[query_financial_data],
    llm=llm,
    system_prompt="""
You are a financial analysis assistant with access to Adobe's 2021-2023 data.

For EVERY user question:
1. Break the question into a minimum number of atomic sub-questions that the tools can answer directly.
   Each atomic question for example must involve exactly one metric and one year.
   Never invoke a tool with a non-atomic question.
2. Call query_financial_data once per atomic question.
3. After receiving all tool results, synthesize them into a concise, well-structured answer for the user (include comparisons, trends, etc.).
4. NEVER call a tool with a question that covers multiple years or multiple metrics.
"""
)

In [ ]:
# test the retrieval capacities
from llama_index.core.agent.workflow import ToolCall, ToolCallResult, AgentStream

handler = query_agent.run("Considering the competition Adobe faces across the Digital Media and Digital Experience segments, how does its end-to-end ecosystem (including Adobe Stock, Behance, Experience Platform, and Sensei) serve as a competitive moat against point-solution competitors like Figma, Shopify, and Google Analytics in 2023? Has that changed since 2021?")
async for ev in handler.stream_events():
  if isinstance(ev, AgentStream):
    print(ev.delta, end="")
  elif isinstance(ev, ToolCallResult):
    print(f"\n**Got Result from `{ev.tool_name}`**")
    print(f"\n\n{ev.tool_output}\n\n")
  elif isinstance(ev, ToolCall):
    print(f"\nCalling tool: {ev.tool_name} with kwargs: {ev.tool_kwargs}")


response = await handler

To answer your question, I will break it down into atomic sub-questions focused on Adobe’s financial performance, as this is the data I can directly access:

**Atomic sub-questions:**
1. What was Adobe’s revenue from the Digital Media segment in 2021?
2. What was Adobe’s revenue from the Digital Media segment in 2023?
3. What was Adobe’s revenue from the Digital Experience segment in 2021?
4. What was Adobe’s revenue from the Digital Experience segment in 2023?

These metrics will help us assess whether Adobe’s end-to-end ecosystem has strengthened its competitive moat, as reflected in its segment growth over time.

I will now retrieve the relevant data.
Calling tool: query_financial_data with kwargs: {'atomic_question': 'What was Adobe’s revenue from the Digital Media segment in 2021?'}

Calling tool: query_financial_data with kwargs: {'atomic_question': 'What was Adobe’s revenue from the Digital Media segment in 2023?'}

Calling tool: query_financial_data with kwargs: {'atomic_questi

## Building an Multi-Agent Report Generation Chatbot

Now, we have an agent capable of gather research to answer a question. With this, we can expand the scope to include full-blown agentic report generation.

This chatbot will be able to
1. Use tool calls to gather research
2. Use tool calls to record the key notes from research
3. Carry on casual Q/A as any other assistant

### Report Structure

First, let's define a data-structure for our report.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Tuple
import pandas as pd
from IPython.display import display, Markdown


class TextBlock(BaseModel):
    """Text block."""

    text: str = Field(..., description="The text for this block.")


class TableBlock(BaseModel):
    """Image block."""

    caption: str = Field(..., description="Caption of the table.")
    col_names: List[str] = Field(..., description="Names of the columns.")
    rows: List[Tuple] = Field(
        ...,
        description=(
            "List of rows. Each row is a data entry tuple, "
            "where each element of the tuple corresponds positionally to the column name."
        )
    )

    def to_df(self) -> pd.DataFrame:
        """To dataframe."""
        df = pd.DataFrame(self.rows, columns=self.col_names)
        df.style.set_caption(self.caption)
        return df


class ReportOutput(BaseModel):
    """Data model for a report.

    Can contain a mix of text and table blocks. Use table blocks to present any quantitative metrics and comparisons.

    """

    blocks: List[TextBlock | TableBlock] = Field(
        ..., description="A list of text and table blocks."
    )

    def render(self) -> None:
        """Render as formatted text within a jupyter notebook."""
        for b in self.blocks:
            if isinstance(b, TextBlock):
                display(Markdown(b.text))
            else:
                display(b.to_df())

With the report structure defined, we can now develop several tools that we can use for our agent to aide in its ability to generate a report.

### Agent State Definition

Throughout the report generation, the agent and tools will have access to some state dict.

In [ ]:
state_dict = {
    "notes": [],
    "report": None,
    "report_schema": ReportOutput.model_json_schema(),
}


### Agent Tools Definition

With that state in mind, lets explore the tools our agent has access to!

1. Research + Evaluation Tool

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import Annotated

from llama_index.core.tools import QueryEngineTool
from llama_index.core.workflow import Context
from llama_index.llms.openai import OpenAIResponses
from llama_index.core.llms import ChatMessage


# evaluation schema
class InfoEvaluation(BaseModel):
  relevance: int = Field(
      description="Relevancy of the information to the user's query, expressed as an integer between 0 and 100",
      ge=0,
      le=100,
  )
  coherence: int = Field(
      description="Coherence of the information to the user's query, expressed as an integer between 0 and 100",
      ge=0,
      le=100,
  )
  reasons: str = Field(
      description="Reasons for the given evaluation"
  )

judge_llm = OpenAIResponses(model="gpt-4.1-mini").as_structured_llm(InfoEvaluation)

# evaluation function
async def evaluate_information(
  original_user_query: Annotated[str, "Original query from the user"],
  retrieved_information: Annotated[str, "Information retrieved starting from the user's query"]
) -> str:
  messages = [ChatMessage(role="user", content=f"This is my original query:\n\n'''\n{original_user_query}\n'''\n\nAnd this is the inforamtion that was gathered to respond to my query:\n\n'''\n{retrieved_information}\n'''\n\nCan you please evaluate this information for relevance and coherence, and give me then reasons for your evaluation?")]
  response = await judge_llm.achat(messages)
  response_json = json.loads(response.message.content)
  return f"The retrieved information is {response_json['relevance']}% relevant and {response_json['coherence']}% coherent to the user's original query.\nHere are the reasons for this evaluation: {response_json['reasons']}"


## retrieve + evaluation
async def generate_research_information(
  ctx: Context,
  query: Annotated[str, "The query/prompt used to generate report notes and research."]
) -> str:
  """Useful for gathering notes and research related to the input query."""
  query_agent = FunctionAgent(
      tools=[query_financial_data],
      llm=OpenAI(model="gpt-4.1"),
      system_prompt="""You are a financial analysis assistant with access to Adobe's 2021-2023 data.

For EVERY user question:
1. Break the question into a minimum number of atomic sub-questions that the tools can answer directly.
  Each atomic question for example must involve exactly one metric and one year.
  Never invoke a tool with a non-atomic question.
2. Call get_financial_metric once per atomic question.
3. After receiving all tool results, synthesize them into a concise, well-structured answer for the user (include comparisons, trends, etc.).
4. NEVER call a tool with a question that covers multiple years or multiple metrics.
"""
  )
  response = await query_agent.run(query)
  response_str = str(response)

  evaluation = await evaluate_information(query, response_str)
  return f"Generated information: {response_str}\n\nEvaluation of Generated Information: {evaluation}"

2. Letting the agent save notes from the generated information

In [ ]:
# record notes
async def save_notes(
  ctx: Context,
  notes: Annotated[str, "Notes derived from the generated research"]
):
  """Tool useful to record the passed in notes into the current state."""
  current_state = await ctx.get("state", default=[])
  current_state["notes"].append(notes)
  await ctx.set("state", current_state)
  return "Notes recorded."

3. Writing a report draft using the generated information

In [ ]:
# record report
async def write_report(
  ctx: Context,
) -> str:
  """Tool useful to write a report based on the current notes and save it into the state."""

  # Assemble pieces of the context
  current_state = await ctx.get("state")
  notes = current_state.get("notes", [])
  if not notes:
    return "Please record some research notes before writing a report."

  notes_str = "\n\n".join(notes)
  prompt = """Here is the current chat history so far:
{chat_history}

Here is some research notes that will be helpful when writing a report:
{notes}

Given these notes and chat history as context, write a report.
"""

  # Condense the current chat history into a string
  memory = await ctx.get("memory")
  chat_history = await memory.aget()
  chat_history_str = ""
  for msg in chat_history:
    chat_history_str += f"<msg role='{msg.role.value}'>{msg.content}</msg>\n"

  # Get the report using a structured llm
  report_llm = OpenAIResponses(model="o3-mini").as_structured_llm(ReportOutput)
  resp = await report_llm.acomplete(
    prompt.format(notes=notes_str, chat_history=chat_history_str)
  )

  report = resp.raw
  current_state["report"] = report.model_dump()
  await ctx.set("state", current_state)

  return "Report written and reviewed."

### Putting it all Together

Now we can assemble our tools into a single agent to create our report-generating chatbot.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow

root_agent = FunctionAgent(
    tools=[generate_research_information, write_report, save_notes],
    llm=OpenAI(model="o3-mini"),
    system_prompt="""You are a financial analysis assistant with access to Adobe's 2021-2023 data.

While you are a generally helpful assistant, your typical flow of actions will generally be
1. Generate some research that will help the user when they ask for a report
2. Save some notes from that research that will help you write a report
3. Write a report based on the notes
"""
)

report_agent = AgentWorkflow(
    agents=[root_agent],
    initial_state={**state_dict},
)

ctx = Context(report_agent)

Let's try out our multi-agent system with a query!

In [ ]:
from llama_index.core.agent.workflow import ToolCall, ToolCallResult, AgentStream

async def run_agent(q: str, agent: AgentWorkflow, ctx: Context) -> None:
  handler = agent.run(q, ctx=ctx)
  async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
      tool_call_str = f"\n**Result from `{ev.tool_name}`**:\n\n{ev.tool_output.content}\n\n"
      display(Markdown(tool_call_str))
    elif isinstance(ev, ToolCall):
      tool_call_str = f"\n### Calling tool: `{ev.tool_name}`\n\n```json\n{ev.tool_kwargs}\n```\n\n"
      display(Markdown(tool_call_str))
    elif isinstance(ev, AgentStream):
      print(ev.delta, end="")

  response = await handler
  display(Markdown(str(response)))

response = await run_agent(
  "Tell me about the top-level assets and liabilities for Adobe from 2021, 2022, and 2023. Are they changing for better or worse?",
  report_agent,
  ctx,
)


### Calling tool: `generate_research_information`

```json
{'query': 'top-level assets and liabilities for Adobe from 2021, 2022, and 2023, and whether they are changing for better or worse'}
```




**Result from `generate_research_information`**:

Generated information: Here are Adobe's top-level assets and liabilities for 2021, 2022, and 2023:

Total Assets:
- 2021: $27,241 million
- 2022: $27,165 million
- 2023: $29,779 million

Total Liabilities:
- 2021: $12,444 million
- 2022: $13,114 million
- 2023: $13,261 million

Analysis:
- Assets slightly decreased from 2021 to 2022, then increased significantly in 2023.
- Liabilities have steadily increased each year, but the growth from 2022 to 2023 was modest compared to the jump in assets.

Overall, Adobe's financial position improved in 2023, with assets growing faster than liabilities, indicating a stronger balance sheet.

Evaluation of Generated Information: The retrieved information is 95% relevant and 90% coherent to the user's original query.
Here are the reasons for this evaluation: The information directly addresses the user's query by providing top-level assets and liabilities for Adobe for the years 2021, 2022, and 2023. It also includes an analysis of whether these figures are changing for better or worse, which aligns well with the user's request. The data is clear and structured, making it easy to understand the trends over the three years. The slight deduction in coherence is due to the briefness of the analysis, which could be expanded with more detailed financial context or implications, but overall it is coherent and relevant to the query.




### Calling tool: `save_notes`

```json
{'notes': "Research Notes on Adobe's top-level assets and liabilities for 2021-2023:\n\n1. Total Assets:\n   - 2021: $27,241 million\n   - 2022: $27,165 million\n   - 2023: $29,779 million\n\n2. Total Liabilities:\n   - 2021: $12,444 million\n   - 2022: $13,114 million\n   - 2023: $13,261 million\n\nAnalysis:\n- Assets: There was a slight decrease from 2021 to 2022, but a significant jump in 2023, indicating an overall positive trend in asset growth.\n- Liabilities: They increased steadily each year, with a modest rise between 2022 and 2023.\n\nConclusion:\n- The improvement in assets from 2022 to 2023, outpacing the growth in liabilities, suggests that Adobe's financial position is strengthening overall."}
```




**Result from `save_notes`**:

Notes recorded.




### Calling tool: `write_report`

```json
{'blocks': [{'text': "This report examines Adobe's top-level assets and liabilities over the years 2021, 2022, and 2023. The data shows a slight decrease in assets from 2021 to 2022, but a significant increase in 2023, while liabilities have shown a steady increase year over year. The rate of asset growth in 2023 has outpaced the increase in liabilities, suggesting an overall improvement in Adobe's financial position."}, {'caption': 'Top-level Assets and Liabilities for Adobe (2021-2023)', 'col_names': ['Year', 'Assets (in million USD)', 'Liabilities (in million USD)'], 'rows': [['2021', '27,241', '12,444'], ['2022', '27,165', '13,114'], ['2023', '29,779', '13,261']]}]}
```




**Result from `write_report`**:

write_report() got an unexpected keyword argument 'blocks'




### Calling tool: `write_report`

```json
{}
```




**Result from `write_report`**:

Report written and reviewed.



Below is a summary of Adobe’s top-level assets and liabilities for 2021, 2022, and 2023, along with a brief analysis on whether the trends indicate an improvement:

Assets and Liabilities Summary:
• 2021: Assets stood at approximately $27,241 million while Liabilities were about $12,444 million.
• 2022: Assets experienced a slight decrease to around $27,165 million, and Liabilities increased to about $13,114 million.
• 2023: There was a significant rebound in assets rising to roughly $29,779 million. Liabilities also rose to around $13,261 million, but the asset increase outpaced the liabilities growth.

Analysis:
While liabilities have shown a consistent upward trend over the three years, the significant jump in asset value in 2023 is a positive signal. The stronger growth in assets relative to liabilities suggests that Adobe’s overall balance sheet has improved, indicating a more robust financial position in 2023 compared to the prior years.

In summary, Adobe appears to be strengthe

Below is a summary of Adobe’s top-level assets and liabilities for 2021, 2022, and 2023, along with a brief analysis on whether the trends indicate an improvement:

Assets and Liabilities Summary:
• 2021: Assets stood at approximately $27,241 million while Liabilities were about $12,444 million.
• 2022: Assets experienced a slight decrease to around $27,165 million, and Liabilities increased to about $13,114 million.
• 2023: There was a significant rebound in assets rising to roughly $29,779 million. Liabilities also rose to around $13,261 million, but the asset increase outpaced the liabilities growth.

Analysis:
While liabilities have shown a consistent upward trend over the three years, the significant jump in asset value in 2023 is a positive signal. The stronger growth in assets relative to liabilities suggests that Adobe’s overall balance sheet has improved, indicating a more robust financial position in 2023 compared to the prior years.

In summary, Adobe appears to be strengthening its financial health, particularly in 2023, by boosting its asset base while managing liabilities at a relatively controlled pace.

In [ ]:
state = await ctx.get("state")
report = ReportOutput.model_validate(state["report"])
report.render()

Adobe's financial performance from 2021 to 2023 shows an overall strengthening position. The company's total assets saw a slight decline between 2021 and 2022 but experienced a significant jump in 2023. In contrast, total liabilities have steadily increased each year, albeit at a modest pace from 2022 to 2023. This suggests that the impressive asset growth in 2023 outweighs the incremental rise in liabilities, signaling an overall positive trend.

,Year,Total Assets (million USD),Total Liabilities (million USD)
0,2021,27241,12444
1,2022,27165,13114
2,2023,29779,13261


In summary, while Adobe's liabilities have been increasing steadily over the period, the significant rise in assets in 2023 suggests that the company's overall financial health is improving. This positive shift implies that Adobe's investments and strategies may be yielding firm asset growth, thereby strengthening its financial position despite the rising liabilities.